# mart_user_weekly_analysis 생성

Staging View 없이 원본 테이블을 결합해 유저 1명 × 주차 1행의 분석 Mart를 생성합니다.

In [15]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
MAX_BYTES = 1100 * 1024**2

client = bigquery.Client(project=PROJECT_ID, location="asia-northeast3")

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Mart 생성 쿼리

In [13]:
mart_select_sql = f"""
WITH friend_edges AS (
    SELECT send_user_id AS user_id, receive_user_id AS friend_id
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_friendrequest`
    WHERE status = 'A'

    UNION DISTINCT

    SELECT receive_user_id, send_user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_friendrequest`
    WHERE status = 'A'
),
friend_counts AS (
    SELECT user_id, COUNT(DISTINCT friend_id) AS friend_count
    FROM friend_edges
    GROUP BY user_id
),
session_user AS (
    SELECT
        hp.session_id,
        ANY_VALUE(SAFE_CAST(hp.user_id AS INT64)) AS user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_properties` AS hp
    WHERE hp.session_id IS NOT NULL
      AND TRIM(hp.session_id) != ''
      AND SAFE_CAST(hp.user_id AS INT64) IS NOT NULL
    GROUP BY hp.session_id
    HAVING COUNT(DISTINCT hp.user_id) = 1
),
weekly_activity AS (
    SELECT
        s.user_id,
        DATE_TRUNC(DATE(e.event_datetime, 'Asia/Seoul'), WEEK(MONDAY)) AS week_start,
        COUNT(DISTINCT DATE(e.event_datetime, 'Asia/Seoul')) AS active_days,
        COUNT(*) AS event_count,
        COUNT(DISTINCT e.session_id) AS session_count,
        COUNTIF(e.event_key = 'click_question_start') AS question_start_count,
        COUNTIF(e.event_key = 'complete_question') AS question_complete_count,
        COUNTIF(e.event_key = 'skip_question') AS question_skip_count,
        COUNTIF(e.event_key = 'click_question_open') AS received_vote_open_count,
        COUNTIF(e.event_key = 'view_timeline_tap') AS timeline_view_count,
        COUNTIF(e.event_key = 'click_attendance') AS attendance_click_count,
        COUNTIF(e.event_key = 'complete_purchase') AS purchase_count
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
    JOIN session_user AS s USING (session_id)
    WHERE DATE(e.event_datetime, 'Asia/Seoul') BETWEEN '2023-07-24' AND '2023-08-10'
    GROUP BY s.user_id, week_start
),
vote_pairs AS (
    SELECT
        DATE_TRUNC(DATE(created_at, 'Asia/Seoul'), WEEK(MONDAY)) AS week_start,
        user_id AS voter_user_id,
        chosen_user_id AS receiver_user_id,
        COUNT(*) AS pair_votes
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_userquestionrecord`
    WHERE DATE(created_at, 'Asia/Seoul') BETWEEN '2023-07-24' AND '2023-08-10'
    GROUP BY week_start, voter_user_id, receiver_user_id
),
votes_given AS (
    SELECT
        week_start,
        voter_user_id AS user_id,
        SUM(pair_votes) AS votes_given,
        COUNT(*) AS unique_receivers
    FROM vote_pairs
    GROUP BY week_start, user_id
),
votes_received AS (
    SELECT
        week_start,
        receiver_user_id AS user_id,
        SUM(pair_votes) AS votes_received,
        COUNT(*) AS unique_voters,
        MAX(pair_votes) AS top_voter_votes
    FROM vote_pairs
    GROUP BY week_start, user_id
)
SELECT
    w.user_id,
    w.week_start,
    DATE(u.created_at, 'Asia/Seoul') AS signup_date,
    u.gender,
    u.group_id,
    gp.school_id,
    gp.grade,
    gp.class_num,
    sc.school_type,
    COALESCE(f.friend_count, 0) AS friend_count,
    w.active_days,
    w.event_count,
    w.session_count,
    w.question_start_count,
    w.question_complete_count,
    w.question_skip_count,
    w.received_vote_open_count,
    w.timeline_view_count,
    w.attendance_click_count,
    w.purchase_count,
    COALESCE(g.votes_given, 0) AS votes_given,
    COALESCE(g.unique_receivers, 0) AS unique_receivers,
    COALESCE(r.votes_received, 0) AS votes_received,
    COALESCE(r.unique_voters, 0) AS unique_voters,
    COALESCE(r.top_voter_votes, 0) AS top_voter_votes,
    SAFE_DIVIDE(r.top_voter_votes, r.votes_received) AS received_concentration,
    CASE
        WHEN u.group_id IS NULL THEN 0
        ELSE COUNT(*) OVER (PARTITION BY w.week_start, u.group_id) - 1
    END AS active_classmates,
    IF(DATE_ADD(w.week_start, INTERVAL 13 DAY) <= DATE '2023-08-10', 1, 0) AS is_retention_eligible,
    CASE
        WHEN DATE_ADD(w.week_start, INTERVAL 13 DAY) > DATE '2023-08-10' THEN NULL
        WHEN n.user_id IS NOT NULL THEN 1
        ELSE 0
    END AS next_week_retained
FROM weekly_activity AS w
LEFT JOIN weekly_activity AS n
    ON w.user_id = n.user_id
   AND n.week_start = DATE_ADD(w.week_start, INTERVAL 7 DAY)
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_user` AS u ON w.user_id = u.id
LEFT JOIN friend_counts AS f ON w.user_id = f.user_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_group` AS gp ON u.group_id = gp.id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_school` AS sc ON gp.school_id = sc.id
LEFT JOIN votes_given AS g
    ON w.user_id = g.user_id
   AND w.week_start = g.week_start
LEFT JOIN votes_received AS r
    ON w.user_id = r.user_id
   AND w.week_start = r.week_start
"""

## 예상 처리량 확인

In [16]:
dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
estimated = client.query(mart_select_sql, job_config=dry_config).total_bytes_processed

print(f"예상 처리량: {estimated / 1024**2:,.2f} MiB")
if estimated > MAX_BYTES:
    raise ValueError("1 GiB 상한을 초과했습니다.")

예상 처리량: 1,042.17 MiB


## Mart 생성

In [17]:
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.mart_user_weekly_analysis`
PARTITION BY week_start
CLUSTER BY user_id, group_id AS
{mart_select_sql}
"""

config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
client.query(sql, job_config=config).result()
print("mart_user_weekly_analysis 생성 완료")

mart_user_weekly_analysis 생성 완료


## 최종 검증

In [20]:
validation_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNTIF(is_retention_eligible = 1) AS eligible_rows,
    COUNTIF(next_week_retained = 1) AS retained_rows
FROM `{PROJECT_ID}.{DATASET_ID}.mart_user_weekly_analysis`
"""

config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES
)

validation_df = client.query(
    validation_sql,
    job_config=config
).to_dataframe(create_bqstorage_client=False)

validation_df

,total_rows,unique_users,eligible_rows,retained_rows
0,273261,188235,117741,44945
